In [ ]:
!pip install evaluate
!pip install sacrebleu
!pip install unbabel-comet

!pip install --upgrade protobuf

# **PREPROCESAMIENTO**

Se usa los archivos train.csv, val.csv y test.csv para el procesamiento del modelo

In [ ]:
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/train.csv"



print(os.listdir("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine"))

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

In [ ]:
#Limpiar texto

import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

df.head()

,source,target,source_clean,target_clean
0,gike. netlu wagaga.,no. veo al guapo.,gike netlu wagaga,no veo al guapo
1,kagwuru wontapa.,vamos a ver la flor.,kagwuru wontapa,vamos a ver la flor
2,ga wa netatkalo yawo.,y vi al pelejo.,ga wa netatkalo yawo,y vi al pelejo
3,wane gyonakya.,ustedes cultivan allá.,wane gyonakya,ustedes cultivan allá
4,"kapiripa neta, ga wa kolyo, ga wa tora.","veo boquichico, y otro pescado, y corbina.",kapiripa neta ga wa kolyo ga wa tora,veo boquichico y otro pescado y corbina


In [ ]:
##DEFINICIÓN DE SILABIFICADOR

VOCALES = "aeiou"
CONSONANTES_COMPUESTAS = ["ch", "sh", "ts"]
SUFIJOS = ["klu", "ru", "lu", "chri", "xlu"]

def dividir_palabra(palabra):
    silabas = []
    actual = ""
    i = 0

    while i < len(palabra):
        if i + 1 < len(palabra):
            par = palabra[i:i+2]
            if par in CONSONANTES_COMPUESTAS:
                actual += par
                i += 2
                continue

        letra = palabra[i]
        actual += letra

        if letra in VOCALES:
            silabas.append(actual)
            actual = ""

        i += 1

    if actual:
        silabas.append(actual)

    return silabas


def silabificar_yine(texto):
    palabras = texto.lower().split()
    resultado = []

    for palabra in palabras:
        if len(palabra) <= 4:
            resultado.append(palabra)
            continue

        partes = palabra.split("-")

        for parte in partes:
            sufijo_encontrado = False

            for sufijo in SUFIJOS:
                if parte.endswith(sufijo) and len(parte) > len(sufijo):
                    raiz = parte[:-len(sufijo)]
                    resultado.extend(dividir_palabra(raiz))
                    resultado.append(sufijo)
                    sufijo_encontrado = True
                    break

            if not sufijo_encontrado:
                resultado.extend(dividir_palabra(parte))

    return resultado

In [ ]:
df["source_silabificado"] = df["source_clean"].apply(
    lambda x: " ".join(silabificar_yine(x))
)

In [ ]:
df_model = df[["source_silabificado", "target_clean"]].rename(
    columns={
        "source_silabificado": "source",
        "target_clean": "target"
    }
)


from datasets import Dataset
dataset = Dataset.from_pandas(df_model)

df_model.head()

,source,target
0,gike ne t lu wa ga ga,no veo al guapo
1,ka gwu ru wo nta pa,vamos a ver la flor
2,ga wa ne ta tka lo yawo,y vi al pelejo
3,wane gyo na kya,ustedes cultivan allá
4,ka pi ri pa neta ga wa ko lyo ga wa tora,veo boquichico y otro pescado y corbina


In [ ]:
#Tokenización

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
##Ejemplo para verificar
example1 = dataset[0]["source"]
example2 = dataset[0]["target"]

print("Texto usado por el modelo:")
print(example1)
print(example2)

tokens = tokenizer(example1)
print(tokens["input_ids"])


Texto usado por el modelo:
gike ne t lu wa ga ga
no veo al guapo
[250004, 40130, 13, 108, 808, 3480, 259, 914, 914, 2]


In [ ]:
##Ejemplo de ambas lenguas

source_text = dataset[0]["source"]
target_text = dataset[0]["target"]

source_tokens = tokenizer(source_text)
target_tokens = tokenizer(target_text)

print("SOURCE IDS:", source_tokens["input_ids"])
print("TARGET IDS:", target_tokens["input_ids"])

SOURCE IDS: [250004, 40130, 13, 108, 808, 3480, 259, 914, 914, 2]
TARGET IDS: [250004, 110, 173, 31, 144, 42348, 771, 2]


In [ ]:
#CREACION DE FUNCION PARA TOKENIZAR TODO EL DATASET

def tokenizar_ejemplo(example):

    # texto de entrada (Yine silabificado)
    source = example["source"]

    # texto objetivo (español)
    target = example["target"]

    # tokenizamos input
    source_tokens = tokenizer(
        source,
        truncation=True,
        padding="max_length",
        max_length=64
    )

    # tokenizamos target
    target_tokens = tokenizer(
        target,
        truncation=True,
        padding="max_length",
        max_length=64
    )

    return {
        "input_ids": source_tokens["input_ids"],
        "attention_mask": source_tokens["attention_mask"],
        "labels": target_tokens["input_ids"]
    }

In [ ]:
#Aplicamos la funcion pta tokenizar todo el dataset

tokenized_dataset = dataset.map(tokenizar_ejemplo)

Map:   0%|          | 0/372 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset[0])

{'source': 'gike ne t lu wa ga ga', 'target': 'no veo al guapo', 'input_ids': [250004, 40130, 13, 108, 808, 3480, 259, 914, 914, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [250004, 110, 173, 31, 144, 42348, 771, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
##Ejecutar solo una vez
tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])


In [ ]:
print(tokenized_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 372
})


In [ ]:

tokenized_dataset.set_format("torch")

In [ ]:

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)

In [ ]:
sample = tokenized_dataset[0]

# agregamos batch dimension
import torch

inputs = {
    "input_ids": sample["input_ids"].unsqueeze(0),
    "attention_mask": sample["attention_mask"].unsqueeze(0),
    "labels": sample["labels"].unsqueeze(0)
}

outputs = model(**inputs)
print(outputs)

Seq2SeqLMOutput(loss=tensor(12.2597, grad_fn=<NllLossBackward0>), logits=tensor([[[ -1.4462,  -1.3631,   1.1101,  ...,   0.2823,   1.8314,  -1.1297],
         [ -1.4362,  -1.4717,  -0.8795,  ...,  -4.5216,  -3.8591,  -1.0731],
         [  2.8824,   2.8770,  11.6766,  ...,   3.0192,   2.0829,   2.9485],
         ...,
         [-16.7256, -16.3754,  -7.5079,  ..., -24.4205, -21.2073, -16.8682],
         [-17.0141, -16.6374,  -7.5609,  ..., -24.9214, -21.7581, -17.1656],
         [-16.8089, -16.4274,  -7.2376,  ..., -24.3749, -21.0641, -16.9481]]],
       grad_fn=<AddBackward0>), past_key_values=None, decoder_hidden_states=None, decoder_attentions=None, cross_attentions=None, encoder_last_hidden_state=tensor([[[ 9.1921e-03, -5.9220e-03,  2.2743e-04,  ..., -2.8671e-02,
          -4.3355e-03,  6.2380e-03],
         [ 1.1472e-01,  8.7876e-01,  3.3790e-02,  ..., -1.6706e+00,
           1.1070e-02, -5.8291e-01],
         [-2.0840e-01, -6.1059e-02, -6.3757e-02,  ..., -7.8188e-01,
          -6.18

In [ ]:
##Código de entrenamiento (mBART50)

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",              # dónde guarda modelo
    per_device_train_batch_size=1,       # cuántos ejemplos por paso
    num_train_epochs=3,                  # cuántas veces ve el dataset
    logging_dir="./logs",                # logs
    save_strategy="epoch",               # guarda cada epoch
)

In [ ]:
#Creación del Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine"))

['Relatos', 'Historias', 'corpus.csv', 'test.csv', 'train.csv', 'val.csv', '1. Limpieza de datos.ipynb', '2. Entrenamiento.ipynb', 'modelo_mbART_yine']


In [ ]:
print(os.listdir("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine"))

['config.json', 'generation_config.json', 'model.safetensors', 'training_args.bin']


In [ ]:
#Entrenar

##SOLO EL PRIMER ENTRENAMIENTO
#trainer.train()

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# modelo entrenado
model = AutoModelForSeq2SeqLM.from_pretrained(
    "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine"
)

# tokenizer ORIGINAL (no el tuyo)
tokenizer = AutoTokenizer.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)

In [ ]:
#Guardar modelo

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_yine")
tokenizer.save_pretrained("/content/drive/MyDrive/Entrenamiento modelos NMT/Yine/modelo_mbART_yine")

('/content/drive/MyDrive/Entrenamiento modelos NMT/Yine/modelo_mbART_yine/tokenizer_config.json',
 '/content/drive/MyDrive/Entrenamiento modelos NMT/Yine/modelo_mbART_yine/special_tokens_map.json',
 '/content/drive/MyDrive/Entrenamiento modelos NMT/Yine/modelo_mbART_yine/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Entrenamiento modelos NMT/Yine/modelo_mbART_yine/added_tokens.json',
 '/content/drive/MyDrive/Entrenamiento modelos NMT/Yine/modelo_mbART_yine/tokenizer.json')

In [ ]:
##Creación de función de traducción

def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_length=50)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

***PRUEBA CON TEST.CSV***

In [ ]:
#Cargar el CSV

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

#Limpiar el texto
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)

#Aplicar el silabificador
df_val["source_silabificado"] = df_val["source_clean"].apply(
    lambda x: " ".join(silabificar_yine(x))
)

#Crear Dataset
df_val_model = df_val[["source_silabificado", "target_clean"]].rename(
    columns={
        "source_silabificado": "source",
        "target_clean": "target"
    }
)

from datasets import Dataset
val_dataset = Dataset.from_pandas(df_val_model)



In [ ]:
#Probar ejemplos

for i in range(5):
    print("INPUT:", val_dataset[i]["source"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(val_dataset[i]["source"]))
    print("-----")

INPUT: katu ga nu nro ta nu
REAL: con quién va a casarse
PRED: mi tío está enfermo
-----
INPUT: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: mi hermanito está enfermo
-----
INPUT: gi pi klu papa
REAL: mi papá no le tenía miedo
PRED: mi papá no tiene miedo
-----
INPUT: wuya gi pa te wa ta nu tka
REAL: anda no tengas más vergüenza
PRED: vamos a ver al shimbillo
-----
INPUT: gi ge po nro peta
REAL: no has visto al ciempiés
PRED: no tiene miedo de nada
-----


In [ ]:
#Creación de las métricas

##PASO 1 - Preparar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(val_dataset[i]["source"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])  # BLEU necesita lista de listas


In [ ]:
#BLEU

import evaluate

bleu = evaluate.load("bleu")

bleu_result = bleu.compute(predictions=preds, references=refs)

print("BLEU:", bleu_result)

BLEU: {'bleu': 0.0, 'precisions': [0.2097560975609756, 0.08176100628930817, 0.035398230088495575, 0.0], 'brevity_penalty': 0.8851915475249605, 'length_ratio': 0.8913043478260869, 'translation_length': 205, 'reference_length': 230}


In [ ]:
#CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 22.354897300090713, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [ ]:
#COMET

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [ ]:
##COMET - Evaluación

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": val_dataset[i]["source"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|

COMET: 0.5689265624336575


In [ ]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=6,
    learning_rate=3e-5,
    save_strategy="epoch"
)

In [ ]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [ ]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)

model.to("cuda")

In [ ]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.859600
1000,0.177400


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=1116, training_loss=0.4746002339120407, metrics={'train_runtime': 424.3593, 'train_samples_per_second': 2.63, 'train_steps_per_second': 2.63, 'total_flos': 151157301313536.0, 'train_loss': 0.4746002339120407, 'epoch': 3.0})

In [ ]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_v2")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_v2")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_v2/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_v2/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_v2/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_v2/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mbART_v2/tokenizer.json')

In [ ]:
##REINICIO DE LA PRIMERA ITERACION

#Cargar el CSV

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

#Limpiar el texto
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)

#Aplicar el silabificador
df_val["source_silabificado"] = df_val["source_clean"].apply(
    lambda x: " ".join(silabificar_yine(x))
)

#Crear Dataset
df_val_model = df_val[["source_silabificado", "target_clean"]].rename(
    columns={
        "source_silabificado": "source",
        "target_clean": "target"
    }
)

from datasets import Dataset
val_dataset = Dataset.from_pandas(df_val_model)




##--------------------
#Probar ejemplos

for i in range(5):
    print("INPUT:", val_dataset[i]["source"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(val_dataset[i]["source"]))
    print("-----")

INPUT: katu ga nu nro ta nu
REAL: con quién va a casarse
PRED: quién llega a dónde
-----
INPUT: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: mi hermanito come corbina
-----
INPUT: gi pi klu papa
REAL: mi papá no le tenía miedo
PRED: mi papá no tiene miedo
-----
INPUT: wuya gi pa te wa ta nu tka
REAL: anda no tengas más vergüenza
PRED: mi hermanito va a cazar
-----
INPUT: gi ge po nro peta
REAL: no has visto al ciempiés
PRED: no has visto al ciempiés
-----


In [ ]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(val_dataset[i]["source"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [ ]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.17469053486224567,
 'precisions': [0.413953488372093,
  0.23668639053254437,
  0.13821138211382114,
  0.09090909090909091],
 'brevity_penalty': 0.932610680893436,
 'length_ratio': 0.9347826086956522,
 'translation_length': 215,
 'reference_length': 230}

In [ ]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 36.815586817314944, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [ ]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": val_dataset[i]["source"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.6216768136490947


In [ ]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [ ]:
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)

df_test["source_silabificado"] = df_test["source_clean"].apply(
    lambda x: " ".join(silabificar_yine(x))
)

In [ ]:
df_test_model = df_test[["source_silabificado", "target_clean"]].rename(
    columns={"source_silabificado": "source", "target_clean": "target"}
)

from datasets import Dataset
test_dataset = Dataset.from_pandas(df_test_model)

In [ ]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(test_dataset[i]["source"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [ ]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.10873953341648868,
 'precisions': [0.3148148148148148,
  0.16143497757847533,
  0.06818181818181818,
  0.05426356589147287],
 'brevity_penalty': 0.9286029058931802,
 'length_ratio': 0.9310344827586207,
 'translation_length': 270,
 'reference_length': 290}

In [ ]:
##CHRF
chrf.compute(predictions=preds, references=refs)

{'score': 27.81305732549606, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [ ]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": test_dataset[i]["source"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiment

COMET: 0.5778490958061624
